In [4]:
# %% [markdown]
# # Project 4: Diabetes Population Health Management
# ## Notebook 1: Data Acquisition & Preprocessing
# 
# **Dataset:** CDC BRFSS 2023
# - Sample: 445,000+ US adults
# - Variables: 300+ health indicators

# %% [markdown]
# ## 1. Environment Setup

# %%
import pandas as pd
import numpy as np
import pyreadstat
import warnings
warnings.filterwarnings('ignore')

print("✓ Environment initialized")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

# %% [markdown]
# ## 2. Load BRFSS 2023 Data

# %%
data_path = '../data/raw/LLCP2023_TEST.csv'

print(f"Loading: {data_path}")
print("Note: Representative 10K sample demonstrating analytical pipeline")

df_raw = pd.read_csv(data_path)

print(f"\n✓ Data loaded successfully")
print(f"Shape: {df_raw.shape[0]:,} respondents × {df_raw.shape[1]:,} variables")
print(f"Memory: {df_raw.memory_usage(deep=True).sum() / 1e6:.2f} MB")

# %%
# Quick inspection
print("\nFirst few column names:")
print(df_raw.columns[:20].tolist())

# %% [markdown]
# ## 3. Extract Diabetes Variables

# %%
diabetes_vars = [
    # Outcomes
    'DIABETE4', 'PREDIAB2', 'DIABAGE3',
    # Risk factors
    '_BMI5', 'EXERANY2', 'FRUIT2', 'VEGETAB2',
    # Complications
    'CVDINFR4', 'CVDSTRK3', 'CHCKDNY2', 'DIABEYE',
    # Demographics
    '_AGE_G', 'BIRTHSEX', '_RACE', 'INCOME3', 'EDUCA',
    # Geographic
    '_STATE', '_CNTY',
    # Survey weights
    '_LLCPWT', '_STSTR', '_PSU'
]

print(f"Extracting {len(diabetes_vars)} variables")

# Verify all exist
missing = [v for v in diabetes_vars if v not in df_raw.columns]
if missing:
    print(f"⚠ Missing: {missing}")
else:
    print("✓ All variables present")

df_diabetes = df_raw[diabetes_vars].copy()
print(f"\nSubset shape: {df_diabetes.shape[0]:,} × {df_diabetes.shape[1]}")

# %% [markdown]
# ## 4. Handle Missing Values

# %%
# Recode BRFSS missing codes
missing_codes = [7, 9, 77, 99, 777, 999, 7777, 9999]
df_diabetes.replace(missing_codes, np.nan, inplace=True)

print("✓ Missing codes recoded to NaN")

# Missing summary
missing_summary = pd.DataFrame({
    'variable': df_diabetes.columns,
    'n_missing': df_diabetes.isnull().sum(),
    'pct_missing': (df_diabetes.isnull().sum() / len(df_diabetes) * 100).round(2)
})

print("\nTop 10 variables by missingness:")
print(missing_summary.sort_values('pct_missing', ascending=False).head(10).to_string(index=False))

# %% [markdown]
# ## 5. Create Analysis Variables

# %%
# Binary diabetes outcome
df_diabetes['has_diabetes'] = df_diabetes['DIABETE4'].apply(
    lambda x: 1 if x == 1 else (0 if x in [2, 3, 4] else np.nan)
)

print("Diabetes Distribution:")
print(df_diabetes['has_diabetes'].value_counts(dropna=False))
print(f"\nPrevalence (unweighted): {df_diabetes['has_diabetes'].mean():.1%}")

# %%
# Prediabetes
df_diabetes['has_prediabetes'] = df_diabetes['PREDIAB2'].apply(
    lambda x: 1 if x == 1 else (0 if x == 2 else np.nan)
)

print("Prediabetes Distribution:")
print(df_diabetes['has_prediabetes'].value_counts(dropna=False))

# %%
# BMI numeric (convert from 2-decimal format: 2540 = 25.40)
df_diabetes['bmi_numeric'] = df_diabetes['_BMI5'] / 100

def categorize_bmi(bmi):
    if pd.isna(bmi):
        return np.nan
    elif bmi < 18.5:
        return 'Underweight'
    elif bmi < 25.0:
        return 'Normal'
    elif bmi < 30.0:
        return 'Overweight'
    else:
        return 'Obese'

df_diabetes['bmi_category'] = df_diabetes['bmi_numeric'].apply(categorize_bmi)

print("BMI Statistics:")
print(df_diabetes['bmi_numeric'].describe())
print("\nBMI Categories:")
print(df_diabetes['bmi_category'].value_counts())

# %% [markdown]
# ## 6. Data Quality Checks

# %%
# Survey weights
print("Survey Weight Statistics:")
print(df_diabetes['_LLCPWT'].describe())

invalid_weights = (df_diabetes['_LLCPWT'] <= 0).sum()
print(f"\nInvalid weights: {invalid_weights}")

# %%
# Age distribution
print("Age Group Distribution:")
print(df_diabetes['_AGE_G'].value_counts().sort_index())

# %%
# State distribution
print("Top 10 States by Sample Size:")
print(df_diabetes['_STATE'].value_counts().head(10))

# %% [markdown]
# ## 7. Save Processed Data

# %%
output_path = '../data/processed/brfss_2023_diabetes.csv'
df_diabetes.to_csv(output_path, index=False)

print(f"✓ Saved to: {output_path}")
print(f"Shape: {df_diabetes.shape[0]:,} × {df_diabetes.shape[1]}")

# %%
# Summary
print("\n" + "="*60)
print("DATA ACQUISITION COMPLETE")
print("="*60)
print(f"Total respondents: {len(df_diabetes):,}")
print(f"Diabetes cases: {df_diabetes['has_diabetes'].sum():.0f}")
print(f"Diabetes prevalence: {df_diabetes['has_diabetes'].mean():.1%}")
print(f"Mean BMI: {df_diabetes['bmi_numeric'].mean():.1f}")
print("="*60)

# %% [markdown]
# ## Next Steps
# 
# **Notebook 2:** Survey Weighting Implementation
# - Weighted prevalence calculations
# - Design-adjusted confidence intervals
# - Stratified analysis

✓ Environment initialized
pandas: 2.2.0
numpy: 1.26.3
Loading: ../data/raw/LLCP2023_TEST.csv
Note: Representative 10K sample demonstrating analytical pipeline

✓ Data loaded successfully
Shape: 10,000 respondents × 21 variables
Memory: 1.68 MB

First few column names:
['DIABETE4', 'PREDIAB2', 'DIABAGE3', '_BMI5', 'EXERANY2', 'FRUIT2', 'VEGETAB2', 'CVDINFR4', 'CVDSTRK3', 'CHCKDNY2', 'DIABEYE', '_AGE_G', 'BIRTHSEX', '_RACE', 'INCOME3', 'EDUCA', '_STATE', '_CNTY', '_LLCPWT', '_STSTR']
Extracting 21 variables
✓ All variables present

Subset shape: 10,000 × 21
✓ Missing codes recoded to NaN

Top 10 variables by missingness:
variable  n_missing  pct_missing
 INCOME3       1221        12.21
   _RACE       1218        12.18
PREDIAB2        411         4.11
    _PSU        387         3.87
  _STATE        384         3.84
  _STSTR        376         3.76
CVDSTRK3        230         2.30
CHCKDNY2        207         2.07
EXERANY2        206         2.06
 DIABEYE        205         2.05
Diabetes D